In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from src.data.data_loader import get_validated_data, load_eda_params
from src.data.process_data import tfidf_weight_binary
from src.data.data_writer import (
    export_fe_weighted_data,
    export_fe_ohe_data,
    export_fe_params,
)
from src.utils.model_exporter import export_fe_scaler

In [ ]:
# Load data
data = get_validated_data()
data.set_index("appid", inplace=True)
eda_outputs = load_eda_params()
print(eda_outputs)

{'popular_genres': ['Indie', 'Casual', 'Adventure', 'Action', 'Simulation', 'Strategy', 'RPG', 'Free To Play', 'Early Access', 'Sports', 'Racing'], 'popular_categories': ['Single-player', 'Family Sharing', 'Steam Achievements', 'Steam Cloud', 'Full controller support', 'Multi-player', 'Partial Controller Support', 'PvP', 'Co-op'], 'redundant_correlation_features': ['PvP', 'Co-op', 'Family Sharing'], 'no_modelable_columns': ['name', 'release_year', 'release_date', 'genres', 'categories', 'developer', 'publisher']}


In [79]:
# Generate new columns based on existing ones (one hot encoding)
one_hot_genres = data["genres"].str.get_dummies(";").astype("int8")
one_hot_categories = data["categories"].str.get_dummies(";").astype("int8")

In [80]:
# Filter low representation features
one_hot_genres = one_hot_genres.filter(items=eda_outputs["popular_genres"])
one_hot_categories = one_hot_categories.filter(items=eda_outputs["popular_categories"])

# Filter redundant correlation features
one_hot_categories = one_hot_categories.drop(
    columns=eda_outputs["redundant_correlation_features"]
)

one_hot_dataframe = pd.DataFrame(
    pd.concat([one_hot_genres, one_hot_categories], axis=1)
)

In [81]:
# Drop unnecessary columns for the model
data = data.drop(columns=eda_outputs["no_modelable_columns"])

In [82]:
# weighing TF-IDF
OHE_weighted_dataframe = pd.DataFrame(
    tfidf_weight_binary(one_hot_dataframe),
    columns=one_hot_dataframe.columns,
)
OHE_weighted_dataframe.index = data.index
OHE_weighted_dataframe

,Indie,Casual,Adventure,Action,Simulation,Strategy,RPG,Free To Play,Early Access,Sports,Racing,Single-player,Steam Achievements,Steam Cloud,Full controller support,Multi-player,Partial Controller Support
appid,,,,,,,,,,,,,,,,,
3057270,1.351654,0.000000,1.903404,1.934981,0.00000,2.618464,2.647094,0.0,0.000000,0.0,0.0,1.040012,0.000000,0.000000,0.000000,0.0,0.000000
3822840,1.351654,1.752587,0.000000,0.000000,2.51001,2.618464,0.000000,0.0,0.000000,0.0,0.0,1.040012,0.000000,0.000000,0.000000,0.0,0.000000
3216640,1.351654,0.000000,1.903404,0.000000,0.00000,2.618464,0.000000,0.0,0.000000,0.0,0.0,1.040012,1.741833,2.356293,2.504437,0.0,0.000000
2403620,1.351654,0.000000,1.903404,1.934981,0.00000,0.000000,0.000000,0.0,0.000000,0.0,0.0,1.040012,1.741833,2.356293,2.504437,0.0,0.000000
1538040,1.351654,1.752587,1.903404,1.934981,0.00000,0.000000,2.647094,0.0,3.250004,0.0,0.0,1.040012,1.741833,2.356293,2.504437,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3097010,1.351654,1.752587,0.000000,0.000000,2.51001,0.000000,0.000000,0.0,0.000000,0.0,0.0,1.040012,0.000000,0.000000,0.000000,0.0,0.000000
3304930,1.351654,0.000000,0.000000,1.934981,0.00000,0.000000,0.000000,0.0,3.250004,0.0,0.0,1.040012,0.000000,0.000000,0.000000,0.0,0.000000
1461580,0.000000,0.000000,0.000000,0.000000,2.51001,0.000000,0.000000,0.0,0.000000,0.0,0.0,1.040012,0.000000,0.000000,0.000000,0.0,0.000000


In [83]:
# Transform columns selected in log1p and standardize
scaler = StandardScaler()
standardized_data = data[["price", "recommendations"]].copy()
standardized_data = np.log1p(standardized_data)

standardized_data = pd.DataFrame(
    scaler.fit_transform(standardized_data[["price", "recommendations"]]).astype('float32'),
    columns=["price", "recommendations"],
    index=standardized_data.index
)

standardized_data.describe()

,price,recommendations
count,6.399500e+04,6.399500e+04
mean,1.108733e-08,-8.583739e-09
std,1.000008e+00,1.000008e+00
min,-1.560754e+00,-3.660991e-01
25%,-8.822340e-01,-3.660991e-01
50%,2.043256e-01,-3.660991e-01
75%,8.027400e-01,-3.660991e-01
max,5.883887e+00,6.100831e+00


In [84]:
# Concat one hot encoded features with transformed selected features
weighted_data = pd.concat([standardized_data, OHE_weighted_dataframe], axis=1)
ohe_data = pd.concat([standardized_data, one_hot_dataframe], axis=1)

In [ ]:
# Save the final dataset to a CSV file
export_fe_weighted_data(weighted_data)
export_fe_ohe_data(ohe_data)

# Generate outputs
fe_outputs = {
    'genres': one_hot_genres.columns.tolist(),
    'categories': one_hot_categories.columns.to_list(),
}
export_fe_params(fe_outputs)
export_fe_scaler(scaler)